In [ ]:
import pandas as pd
import datetime

BRONZE = '/mnt/c/Users/jortega/code/invalid_emails_ideal_persona_flow/bronze'
RAW_DATA = '/mnt/c/Users/jortega/code/invalid_emails_ideal_persona_flow/raw_data'

# Only the SFDC file has a stray index column. Don't use index_col=0 on the other two.
sfdc_people    = pd.read_csv(f'{BRONZE}/sfdc_people_list.csv', dtype=str, index_col=0)
engagements    = pd.read_csv(f'{BRONZE}/grouped_engagements.csv', dtype=str)
inbound_emails = pd.read_csv(f'{BRONZE}/inbound_reply_activities.csv', dtype=str)
contact_role = pd.read_csv(f'{RAW_DATA}/contact_roles_opportunities.csv', dtype=str)

contact_roles = contact_role.rename(columns={'Contact ID': 'Related Record ID'})
contact_roles['is_contact_role'] = 1

# --- Clean keys ---
sfdc_people['Email'] = sfdc_people['Email'].str.strip().str.lower()

# Engagement IDs are 18 characters, SFDC's are 15. Trim to 15 so they can match.
engagements['Related Record ID'] = engagements['Related Record ID'].str.strip().str[:15]

# --- Collapse inbound replies to one row per email ---
inbound_emails['Created Date'] = pd.to_datetime(inbound_emails['Created Date'], errors='coerce')

inbound_summary = (
    inbound_emails
    .dropna(subset=['Email'])
    .sort_values('Created Date')
    .groupby('Email')
    .agg(
        Last_Reply_Date=('Created Date', 'max'),
        Reply_Count=('Email', 'size'),
        Last_Reply_Subject=('Subject', 'last'),
        Reply_Source=('Source', 'last'),
    )
    .reset_index()
)

# --- Merge: SFDC people + latest engagement (by ID) + inbound replies (by email) ---
merged = (
    sfdc_people
    .merge(engagements, on='Related Record ID', how='left', validate='many_to_one')
    .merge(inbound_summary, on='Email', how='left', validate='many_to_one')
    .merge(contact_role, on='Email', how='left', validate='many_to_one')
)

today = pd.Timestamp.today().normalize()

merged['Engagement_Date']   = pd.to_datetime(merged['Engagement_Date'], errors='coerce')
merged['Last_Reply_Date']   = pd.to_datetime(merged['Last_Reply_Date'], errors='coerce')
merged['Last Meeting Date'] = pd.to_datetime(merged['Last Meeting Date'], errors='coerce')

merged['days_since_engagement']    = (today - merged['Engagement_Date']).dt.days
merged['days_since_inbound_reply'] = (today - merged['Last_Reply_Date']).dt.days
merged['days_since_last_meeting']  = (today - merged['Last Meeting Date']).dt.days

KeyError: 'From Address'

In [ ]:
# merged.to_csv('/mnt/c/Users/jortega/code/invalid_emails_ideal_persona_flow/silver/sfdc_people_with_overall_activity.csv')